# Lab 1 — RAG with ChromaDB  ·  Participant Notebook

Build a Retrieval-Augmented-Generation pipeline over a PDF knowledge base:
**load → split → embed → store → retrieve → grounded answer.**

**How this works**
1. Run the **Setup** cell once.
2. Each task has a `%%task N` code cell — write your code under the `%%task N` line and run it.
   Variables persist between cells (`documents` → `chunks` → `vectorstore` → …).
3. Run the **Score** cell at the bottom any time — it grades every `%%task` cell you've run,
   criterion by criterion, out of 100 (weights: T1 5 · T2 20 · T3 25 · T4 25 · T5 25).

Fixed spec: chunk_size **1000**, overlap **200**, retrieval **k = 3**, refusal sentence exactly
`I could not find the answer in the provided document.`


## Setup — run once


In [ ]:
import sys, os, pathlib
HERE = pathlib.Path.cwd()            # folder holding sandbox.py / grader.py / content/
sys.path.insert(0, str(HERE))

os.environ["SANDBOX_MODE"] = "mock"   # "mock" (no key) | "openai" | "claude"
# os.environ["OPENAI_API_KEY"]   = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

import sandbox
from sandbox import (get_chat_model, get_embeddings, make_vectorstore,
                     load_sample_docs, split_docs, SAMPLE_PDF)
MODE = os.environ["SANDBOX_MODE"]

# ---- %%task magic: stores the cell's source AND runs it -------------------
from IPython.core.magic import register_cell_magic
_TASK_SRC = {}
@register_cell_magic
def task(line, cell):
    _TASK_SRC[int(line.strip())] = cell
    get_ipython().run_cell(cell)

print("MODE =", MODE, "| SAMPLE_PDF exists:", pathlib.Path(SAMPLE_PDF).is_file())


## Task 1 — Load the PDF  (5 marks)
Load `SAMPLE_PDF` with LangChain's PDF loader into `documents`; print the page count.

<details><summary>Hints</summary>

- Loader lives in `langchain_community.document_loaders`, class name starts `PyPDF…`
- `.load()` returns a list of `Document`s, one per page
</details>


In [ ]:
%%task 1
# TODO: from langchain_community.document_loaders import PyPDFLoader
# TODO: loader = PyPDFLoader(SAMPLE_PDF)
# TODO: documents = loader.load()
print("SAMPLE_PDF =", SAMPLE_PDF)
# TODO: print the number of pages


## Task 2 — Split into chunks  (20 marks)
Split `documents` with `RecursiveCharacterTextSplitter`, **chunk_size=1000, chunk_overlap=200**,
into `chunks`; print `len(chunks)`.


In [ ]:
%%task 2
# TODO: from langchain_text_splitters import RecursiveCharacterTextSplitter
chunk_size = 1000
chunk_overlap = 200
# TODO: splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
# TODO: chunks = splitter.split_documents(documents)
# TODO: print('Number of chunks:', len(chunks))


## Task 3 — Embeddings + vector store  (25 marks)
`embeddings = get_embeddings(MODE)`  ·  `vectorstore = make_vectorstore(chunks, embeddings)`  ·
then `vectorstore.similarity_search(query, k=3)` and print the hits.


In [ ]:
%%task 3
# TODO: embeddings = get_embeddings(MODE)
# TODO: vectorstore = make_vectorstore(chunks, embeddings)
query = "How many casual leave days are provided each year?"
# TODO: results = vectorstore.similarity_search(query, k=3)
# TODO: for d in results: print(d.metadata.get('page'), d.page_content[:120])


## Task 4 — Grounded question-answering  (25 marks)
Retriever (**k=3**) + `get_chat_model(MODE)` + a `ChatPromptTemplate` that answers **only** from
`{context}`; if missing, replies exactly `I could not find the answer in the provided document.`
Answer one `{question}`; print the answer and the source pages.


In [ ]:
%%task 4
from langchain_core.prompts import ChatPromptTemplate
# TODO: retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
# TODO: llm = get_chat_model(MODE)
# TODO: prompt = ChatPromptTemplate.from_template( a template string with {context} and {question},
#         instructing: answer ONLY from context, else reply exactly:
#         I could not find the answer in the provided document.
question = "What is the standard notice period for regular full-time employees?"
# TODO: docs = retriever.invoke(question)
# TODO: context = '\n\n'.join(d.page_content for d in docs)
# TODO: resp = llm.invoke(prompt.format_messages(context=context, question=question))
# TODO: print('Answer:', resp.content)
# TODO: print('Pages:', sorted({d.metadata.get('page', '?') for d in docs}))


## Task 5 — Interactive RAG app  (25 marks)
Wrap Task 4 in `def answer(question):` returning `(text, pages)`, then a `while True:` loop that
reads `input()`, prints the answer, and stops on `exit`. Reuse `retriever` / `llm` / `prompt`.


In [ ]:
%%task 5
def answer(question):
    docs = retriever.invoke(question)
    context = '\n\n'.join(d.page_content for d in docs)
    resp = llm.invoke(prompt.format_messages(context=context, question=question))
    pages = sorted({d.metadata.get('page', '?') for d in docs})
    return resp.content.strip(), pages

while True:
    q = input('Question> ').strip()
    if not q:
        continue
    if q.lower() == 'exit':
        break
    text, pages = answer(q)
    print('A:', text, '\npages:', pages)


---
## Score — run this any time
Grades every `%%task` cell you have run so far, using the same rubric as the Streamlit app.


In [ ]:
from grader import grade_task, assemble_source

if not _TASK_SRC:
    print('Run some %%task cells first.')
else:
    order = sorted(_TASK_SRC)
    total = max_total = 0.0
    for tid in order:
        src = assemble_source([_TASK_SRC[i] for i in order if i <= tid], MODE)
        r = grade_task(tid, _TASK_SRC[tid], lab_id="01-rag", accumulated_source=src, mode=MODE)
        total += r['score']; max_total += r['max_score']
        print(f"Task {tid}: {r['score']:.0f}/{r['max_score']}")
        for c in r['criteria']:
            print(f"   {'PASS' if c['passed'] else 'FAIL'}  {c['name']:<26} {c['earned']}/{c['points']}  {c['reason']}")
    print(f"\nTOTAL  {total:.0f} / {max_total:.0f}")


<details><summary>Reference solutions (open only after you've tried)</summary>

See `RAG_Lab.ipynb` in the same folder for a fully-worked version of every task.
</details>
